# Government720

## Training of Federated Dataset

### Importing and Splitting Data

In [1]:
# Access Google Drive for Excel file
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import packages
import pandas as pd
import tensorflow as tf
import numpy as np

In [3]:
# Read in federated client datasets
file_path = '/content/drive/My Drive/G720_FEDERATED.xlsx'  # Finalized dataset
sheets = pd.ExcelFile(file_path).sheet_names

# Define split date
split_date = '2021-01-01'

client_data = {}  # Dictionary to store client datasets
# Open Excel sheet and split data into training and testing per client
for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    # Define training and testing sets based on date split
    train_mask = df["DATE"] < split_date
    test_mask = df["DATE"] >= split_date
    train_df = df[train_mask]
    test_df = df[test_mask]
    # Set DATE to index
    train_df.set_index("DATE", inplace=True)
    test_df.set_index("DATE", inplace=True)
    # Save feature names
    feature_names = list(train_df.drop('GOVT_SATISFACTION', axis=1).columns)
    # Define X_train, y_train, X_test, y_test for each client
    labels_train = train_df['GOVT_SATISFACTION'].values
    features_train = train_df.drop('GOVT_SATISFACTION', axis=1).values
    X_train = features_train
    y_train = labels_train
    labels_test = test_df['GOVT_SATISFACTION'].values
    features_test = test_df.drop('GOVT_SATISFACTION', axis=1).values
    X_test = features_test
    y_test = labels_test
    client_data[sheet] = {'X_train': X_train, 'y_train': y_train, 'X_test': X_test, 'y_test': y_test, 'feature_names': feature_names}

In [4]:
print(client_data) # To verify

{'CLIENT1': {'X_train': array([[ 9.07713402e-02,  4.28571429e-01,  0.00000000e+00, ...,
         6.58336037e-02,  0.00000000e+00,  5.63253420e+00],
       [ 8.83358744e-02,  4.39460188e-01, -6.45062393e-03, ...,
         7.15544939e-02,  3.96384768e-03,  5.60521660e+00],
       [ 6.83087639e-02,  4.61813803e-01,  2.08505325e-02, ...,
         7.07639889e-02, -5.04907004e-03,  5.60910131e+00],
       ...,
       [ 2.71868036e-01,  3.14350154e-01,  6.35025506e-01, ...,
         4.38141108e-02,  7.32617687e-01,  3.54299721e-01],
       [ 2.70880152e-01,  3.27219170e-01,  6.66045050e-01, ...,
         3.32215925e-02,  7.23261049e-01,  3.54220929e-01],
       [ 2.58172910e-01,  3.23932540e-01,  6.68965388e-01, ...,
         2.82055433e-02,  7.26318093e-01,  3.56262531e-01]]), 'y_train': array([0.98412698, 0.96993573, 0.97818637, ..., 0.09899512, 0.07237834,
       0.06976402]), 'X_test': array([[0.26315789, 0.3125    , 0.64705882, ..., 0.02469136, 0.72293987,
        0.35290557],
       [0.

### FL Model Setup

In [5]:
# Import model packages
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Reshape

In [6]:
# Function to create client models using Sequential
def create_client_model(input_dim, embedding_dim=8):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation='relu'),
        Dense(embedding_dim, activation='relu')  # Output an embedding
    ])
    return model

In [7]:
# Function to create server model using LSTM
def create_server_model(num_clients, embedding_dim=8):
    total_input_dim = num_clients * embedding_dim
    model = Sequential([
        Input(shape=(1, total_input_dim)),   # (batch, time_steps=1, features)
        LSTM(50, activation='relu'),
        Dense(1)  # Regression output
    ])
    return model

In [8]:
# Parameters
embedding_dim = 8
epochs = 10
batch_size = 32
learning_rate = 0.001

In [9]:
# Create client models
client_models = {}
for client_id, data in client_data.items():
    input_dim = data['X_train'].shape[1]
    model = create_client_model(input_dim, embedding_dim)
    client_models[client_id] = model

In [10]:
# Create server model
num_clients = len(client_models)
server_model = create_server_model(num_clients, embedding_dim)
print("Server model created.")

Server model created.


In [11]:
# Separate optimizers
client_optimizers = {client_id: Adam(learning_rate) for client_id in client_models}
server_optimizer = Adam(learning_rate)

### FL Model Training w/ Correlation and Explainability Analysis

In [12]:
# Import packages
from scipy.stats import pearsonr
import shap
import matplotlib.pyplot as plt

In [13]:
# Training loop for FL model
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    for i in range(0, 1092, batch_size):
        client_embeddings = []

        with tf.GradientTape(persistent=True) as tape:
            for client_id, model in client_models.items():
                X_batch = client_data[client_id]['X_train'][i:i+batch_size]
                embedding = model(X_batch, training=True)
                client_embeddings.append(embedding)

            # Combine and reshape for LSTM
            combined_embedding = tf.concat(client_embeddings, axis=1)  # (batch_size, total_embedding_dim)
            combined_embedding = tf.expand_dims(combined_embedding, axis=1)  # (batch_size, 1, total_embedding_dim)

            # Labels
            y_true = client_data[list(client_models.keys())[0]]['y_train'][i:i+batch_size]

            # Server forward pass
            y_pred = server_model(combined_embedding, training=True)

            # Loss
            loss = tf.reduce_mean(tf.square(y_true - tf.squeeze(y_pred)))

        # Gradients and updates
        server_grads = tape.gradient(loss, server_model.trainable_variables)
        server_optimizer.apply_gradients(zip(server_grads, server_model.trainable_variables))

        for client_id, model in client_models.items():
            client_grads = tape.gradient(loss, model.trainable_variables)
            client_optimizers[client_id].apply_gradients(zip(client_grads, model.trainable_variables))

        print(f"Batch {i//batch_size + 1}: Loss = {loss.numpy():.4f}")


Epoch 1/10
Batch 1: Loss = 0.0750
Batch 2: Loss = 0.0069
Batch 3: Loss = 0.0224
Batch 4: Loss = 0.0069
Batch 5: Loss = 0.0267
Batch 6: Loss = 0.0154
Batch 7: Loss = 0.0067
Batch 8: Loss = 0.0030
Batch 9: Loss = 0.0081
Batch 10: Loss = 0.0016
Batch 11: Loss = 0.0028
Batch 12: Loss = 0.0035
Batch 13: Loss = 0.0011
Batch 14: Loss = 0.0066
Batch 15: Loss = 0.0088
Batch 16: Loss = 0.0535
Batch 17: Loss = 0.0175
Batch 18: Loss = 0.0064
Batch 19: Loss = 0.0067
Batch 20: Loss = 0.0125
Batch 21: Loss = 0.0457
Batch 22: Loss = 0.0371
Batch 23: Loss = 0.0137
Batch 24: Loss = 0.0137
Batch 25: Loss = 0.0298
Batch 26: Loss = 0.0150
Batch 27: Loss = 0.0076
Batch 28: Loss = 0.0108
Batch 29: Loss = 0.0028
Batch 30: Loss = 0.0091
Batch 31: Loss = 0.0074
Batch 32: Loss = 0.0008
Batch 33: Loss = 0.0077
Batch 34: Loss = 0.1195
Batch 35: Loss = 0.1812

Epoch 2/10
Batch 1: Loss = 0.4810
Batch 2: Loss = 0.2917
Batch 3: Loss = 0.1523
Batch 4: Loss = 0.0080
Batch 5: Loss = 0.0129
Batch 6: Loss = 0.0159
Batch 7

### Model Evaluation

In [15]:
# Loop over batches
for i in range(0, len(list(client_data.values())[0]['X_train']), batch_size):
    print(f"\n=== Batch {i//batch_size + 1} Analysis ===")

    for client_id, model in client_models.items():
        # Slice batch
        X_batch_np = client_data[client_id]['X_train'][i:i+batch_size]
        y_batch_np = client_data[client_id]['y_train'][i:i+batch_size]

        # Skip if batch is empty (e.g., leftover small batch at the end)
        if X_batch_np.shape[0] == 0:
            continue

        # Create DataFrame
        df = pd.DataFrame(X_batch_np, columns=client_data[client_id]['feature_names'])
        df['GOVT_SATISFACTION'] = y_batch_np

        # Calculate correlations
        corr = df.corr()['GOVT_SATISFACTION'].drop('GOVT_SATISFACTION')
        print(f"Client {client_id} correlations:\n{corr}\n")

        # SHAP analysis
        print(f"Generating SHAP values for client {client_id}...")
        explainer = shap.Explainer(model, X_batch_np)
        shap_values = explainer(X_batch_np)

        # Compute mean absolute SHAP values for each feature
        mean_shap_values = np.mean(np.abs(shap_values.values), axis=0)

        # Print the mean SHAP values for each feature
        print(f"Mean SHAP values for client {client_id}:")
        for feature_name, mean_shap_value in zip(client_data[client_id]['feature_names'], mean_shap_values):
            # Handle if mean_shap_value is array-like
            if np.isscalar(mean_shap_value):
                print(f"{feature_name}: {mean_shap_value:.4f}")
            else:
                print(f"{feature_name}: {mean_shap_value[0]:.4f}")

Streaming output truncated to the last 5000 lines.
DOM_TERROR_DEFENDANTS    0.307118
FBI_INVESTIGATIONS       0.721287
FBI_DISRUPTIONS          0.332642
CRIM_JUSTICE_OPINION    -0.618216
CYBERATTACKS            -0.764322
CYBERATTACK_LOSSES       0.714475
ANTIGOV_HATE_GROUPS      0.296594
BLACK_POLICE_KILLINGS    0.675587
BLACK_POLICE_INJURIES    0.718968
Name: GOVT_SATISFACTION, dtype: float64

Generating SHAP values for client CLIENT11...
Mean SHAP values for client CLIENT11:
ACTIVE_SHOOTERS: 0.0001
BACKGROUND_CHECKS: 0.0009
FIREARM_PROD: 0.0126
DOM_VIOLENCE_RATE: 0.0034
INCARCERATED_POP: 0.0010
HOMICIDE_RATE: 0.0003
DOM_TERROR_INCIDENTS: 0.0003
DOM_TERROR_DEFENDANTS: 0.0001
FBI_INVESTIGATIONS: 0.0017
FBI_DISRUPTIONS: 0.0012
CRIM_JUSTICE_OPINION: 0.0001
CYBERATTACKS: 0.0036
CYBERATTACK_LOSSES: 0.0395
ANTIGOV_HATE_GROUPS: 0.0016
BLACK_POLICE_KILLINGS: 0.0044
BLACK_POLICE_INJURIES: 0.0012
Client CLIENT12 correlations:
AGRI_SPENDING    0.259810
FARMS            0.679255
AGRI_SALES      -

In [22]:
# Import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

print("\n--- Evaluating Federated Model ---")

# Prepare test data (hidden states from clients)
client_hidden_test = []

for client_id, (client_name, data) in enumerate(client_data.items()):
    X_test = data['X_test']

    # Client outputs embeddings
    hidden_out = client_models[client_name].predict(X_test, verbose=1)
    client_hidden_test.append(hidden_out)

# Stack client outputs together
server_input_test = np.concatenate(client_hidden_test, axis=-1)  # Shape: (n_test_samples, total_embedding_dim)

# **Expand dims to match LSTM input**: (batch_size, time_steps=1, features)
server_input_test = np.expand_dims(server_input_test, axis=1)

# Server prediction
y_pred = server_model.predict(server_input_test, verbose=1).flatten()

# Ground truth
sample_client = next(iter(client_data.values()))
y_true = sample_client['y_test']

# Metrics
mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mse)

print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")


--- Evaluating Federated Model ---
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Test MSE: 0.0067
Test MAE: 0.0557
Test RMSE: 0.0816


In [17]:
# Function to evaluate FL approach's generalization across all clients
def evaluate_client_generalization(server_model, client_models, client_data):
    print("\n Generalization Check Across Clients:")
    all_errors = []

    for client_id in client_models:
        X_test = client_data[client_id]['X_test']
        y_test = client_data[client_id]['y_test']

        # Forward pass through client model
        client_embedding = client_models[client_id](X_test, training=False)

        # Collect all client embeddings for this batch (just for alignment)
        all_embeddings = []
        for cid in client_models:
            emb = client_models[cid](client_data[cid]['X_test'], training=False)
            all_embeddings.append(emb)

        combined_embedding = tf.concat(all_embeddings, axis=1)
        combined_embedding = tf.expand_dims(combined_embedding, axis=1)

        y_pred = server_model(combined_embedding, training=False)
        mse = tf.reduce_mean(tf.square(y_test - tf.squeeze(y_pred))).numpy()

        all_errors.append(mse)
        print(f"Client {client_id}: MSE = {mse:.4f}")

    std_dev = np.std(all_errors)
    print(f"\n Std Dev of MSE across clients: {std_dev:.4f}")
    return all_errors

In [18]:
# Function to compute fairness across clients
def compute_fairness_metrics(client_errors):
    max_error = max(client_errors)
    min_error = min(client_errors)
    gap = max_error - min_error
    print(f"\n Fairness Gap (Max - Min MSE): {gap:.4f}")
    return gap

In [19]:
# Call functions and print results
client_errors = evaluate_client_generalization(server_model, client_models, client_data)
fairness_gap = compute_fairness_metrics(client_errors)
print(client_errors)
print(fairness_gap)


 Generalization Check Across Clients:
Client CLIENT1: MSE = 0.0067
Client CLIENT2: MSE = 0.0067
Client CLIENT3: MSE = 0.0067
Client CLIENT4: MSE = 0.0067
Client CLIENT5: MSE = 0.0067
Client CLIENT6: MSE = 0.0067
Client CLIENT7: MSE = 0.0067
Client CLIENT8: MSE = 0.0067
Client CLIENT9: MSE = 0.0067
Client CLIENT10: MSE = 0.0067
Client CLIENT11: MSE = 0.0067
Client CLIENT12: MSE = 0.0067

 Std Dev of MSE across clients: 0.0000

 Fairness Gap (Max - Min MSE): 0.0000
[np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086), np.float32(0.0066523086)]
0.0


In [20]:
# End of code 4/22/25
# Edited 4/27/25